# 26 · Pooled training versus pairing (post hoc; plan dated 30 Oct 2026)

Trains the four 5-class designs (DenseNet121, EfficientNet-B3, ResNet50, RETFound; 3 runs each) on **EyePACS and DDR pooled**, with the canonical recipe unchanged, then scores them on the EyePACS fresh set, DDR's official test file and BRSET, and compares one pooled model with pairs of single-source models at the same referral share. The plan (questions, comparisons and predictions) is written to `04_results/plan_pooled_c1.json` **before** any training, and the analysis refuses to run on predictions that predate it.

**Runs end to end:** setup, data, training (resumable), inference, analysis. If the session ends, re-run from the top: finished checkpoints and prediction files are skipped.

**Runtime:** A100, High-RAM, `HF_TOKEN` secret for RETFound. Training is the long part: the three CNN designs (9 models) train first, RETFound (3 models) last. Expect roughly a day of GPU time in total, across several sessions.

**Data rules.** BRSET is used only as a test set. No image, identifier or record-level BRSET value is printed; only aggregates are written.

In [2]:
# Cell 1: environment, plan guard
!pip -q install timm transformers
from google.colab import drive, auth, userdata
drive.mount('/content/drive'); auth.authenticate_user()
import io, os, glob, json, time, zipfile, tarfile, datetime, random, hashlib, itertools
import numpy as np, pandas as pd, torch, torch.nn as nn
from huggingface_hub import login
login(userdata.get('HF_TOKEN'))
from googleapiclient.discovery import build as gbuild
from googleapiclient.http import MediaIoBaseDownload
svc = gbuild('drive', 'v3')
DATA = '/content/drive/MyDrive/stochastic_orthogonality/data'
BASE = f'{DATA}/dr_ensemble_v2'; BR = f'{DATA}/brset'
SPLITS, CKPT, PREDS, RESULTS, AUDIT, DDR_IN = (f'{BASE}/{d}' for d in ('01_splits', '02_checkpoints', '03_predictions', '04_results', '99_audit', '08_ddr'))
OUTD = f'{RESULTS}/revisions_c1'; os.makedirs(OUTD, exist_ok=True)
EYEPACS_SPLIT_IDS = {'train': '1FU30c6vRYMLqX8h9z2F58iEqow1Zx8NQ', 'val': '1Z9tFlAGe0j4RVRgG9-6LnhTrukvFy87y'}
EYEPACS_IMAGE_ZIPS = ['11WspNNHS73OadbW-sqy7BSzcrsBkX1d6', '1zG7rHjm8Ty9LHG4LqMzgnOzrT_90VO9H', '1L88vUNSxWjdGPBquhknZhafcnXXRd3ES',
                      '1-FrpGB74qEGzDvXlA9FhmB2OeeZkKPAN', '1mEIrSGVExZ6zmxnRLxyFab5yB1kH5YBT']
F_SPLITS, F_FRESH_IMG = '13x_CS7N_UasGRSuNYMyD5To2RTYjL-EP', '15mufJLdRXNtZFSHQ_kFHIwmON8ewrPAX'
RETFOUND_REPO = 'iszt/RETFound_mae_meh'; NUM_WORKERS = min(8, os.cpu_count())
def sha256_file(p, chunk=1 << 22):
    h = hashlib.sha256()
    with open(p, 'rb') as f:
        for b in iter(lambda: f.read(chunk), b''): h.update(b)
    return h.hexdigest()
def dl(fid, path):
    with io.FileIO(path, 'wb') as fh:
        d = MediaIoBaseDownload(fh, svc.files().get_media(fileId=fid), chunksize=64 * 1024 * 1024); done = False
        while not done: _, done = d.next_chunk()
def find_id(name, folder):
    r = svc.files().list(q=f"'{folder}' in parents and name='{name}' and trashed=false", fields='files(id)').execute().get('files', [])
    return r[0]['id'] if r else None
def list_folder(folder):
    out, tok = [], None
    while True:
        r = svc.files().list(q=f"'{folder}' in parents and trashed=false", fields='files(id,name),nextPageToken', pageToken=tok, pageSize=1000).execute()
        out += r.get('files', []); tok = r.get('nextPageToken')
        if not tok: break
    return out
PLAN = {
 "title": "Pooled training versus pairing", "written": "2026-10-30",
 "models": "DenseNet121, EfficientNet-B3, ResNet50, RETFound (5-class), 3 runs each, trained on EyePACS v2 train + DDR official train (concatenated, no reweighting); best epoch on EyePACS v2 val + DDR official valid (AUC, referable); canonical recipe; seeds 20261101 + 10 x design index + run",
 "test_sets": {"primary": "BRSET, f = 1.31 x referable prevalence, patient bootstrap", "secondary": ["EyePACS fresh, f = 0.25, patients", "DDR official test, f = 1.31 x referable prevalence in that file, filename groups"]},
 "outcome": "missed severe DR (grade 3 or 4) per 10,000 at the same referral share; referable and PDR secondary; ties at the margin broken at random (expected value)",
 "comparisons_brset_severe": {
   "C1": "pooled single vs EyePACS-only single, matched design and run; prediction: pooled misses fewer",
   "C2": "pooled single (average of 12) vs mixed-source pair (EyePACS design a + DDR design b, a != b, 9 run combinations); prediction: pooled misses fewer",
   "C3": "pair of pooled models (different designs, 9 run combinations) vs pooled average member; prediction: pair misses fewer; vs better member reported without prediction",
   "C4": "pair of pooled models vs EyePACS-only pair (different designs); prediction: pair of pooled misses fewer"},
 "decision": "supported if the 95% patient-bootstrap interval (1,000 resamples) of the difference excludes zero in the predicted direction; contradicted if it excludes zero the other way; otherwise inconclusive",
 "brset_role": "test only; never used for training or model selection"}
PLAN_FILE = f'{RESULTS}/plan_pooled_c1.json'
if not os.path.exists(PLAN_FILE):
    assert not glob.glob(f'{PREDS}/*pooled*'), 'pooled predictions exist without a plan file: stop.'
    PLAN['written_utc'] = datetime.datetime.now(datetime.timezone.utc).isoformat()
    with open(PLAN_FILE, 'w') as fh: json.dump(PLAN, fh, indent=1)
    print('plan written', PLAN_FILE)
PLAN_SHA = sha256_file(PLAN_FILE); PLAN_TIME = os.path.getmtime(PLAN_FILE)
print('plan sha256', PLAN_SHA[:16], '| GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE')

Mounted at /content/drive
plan sha256 a4d0a3178cbc933e | GPU: NVIDIA RTX PRO 6000 Blackwell Server Edition


In [3]:
# Cell 2: roster (hyperparameters copied verbatim from the canonical notebooks)
MODELS = [
  {'name':'densenet121_5class','arch':'densenet121','task':'5class','nc':5,'epochs':15,'bs':32,'lr':1e-4,'wd':1e-4},
  {'name':'efficientnet_b3_5class','arch':'efficientnet_b3','task':'5class','nc':5,'epochs':15,'bs':32,'lr':1e-4,'wd':1e-4},
  {'name':'resnet50_5class','arch':'resnet50','task':'5class','nc':5,'epochs':15,'bs':32,'lr':1e-4,'wd':1e-4},
  {'name':'retfound_5class','arch':'retfound','task':'5class','nc':5,'epochs':15,'bs':8,'lr':5e-5,'wd':0.05,'clip':1.0,'lldecay':0.65},
]
SEEDS = [0, 1, 2]; BASE_SEED = 20261101; IMG_SIZE = 224
def tag(k): return '' if k == 0 else f'_seed{k}'
ORDER = [(mi, cfg, k) for mi, cfg in enumerate(MODELS) if cfg['arch'] != 'retfound' for k in SEEDS] + [(mi, cfg, k) for mi, cfg in enumerate(MODELS) if cfg['arch'] == 'retfound' for k in SEEDS]
print('training order:', [f"{c['name']}{tag(k)}" for _, c, k in ORDER])

training order: ['densenet121_5class', 'densenet121_5class_seed1', 'densenet121_5class_seed2', 'efficientnet_b3_5class', 'efficientnet_b3_5class_seed1', 'efficientnet_b3_5class_seed2', 'resnet50_5class', 'resnet50_5class_seed1', 'resnet50_5class_seed2', 'retfound_5class', 'retfound_5class_seed1', 'retfound_5class_seed2']


In [4]:
# Cell 3: training data (EyePACS v2 train/val + DDR official train/valid) and test images
os.makedirs('/content/img', exist_ok=True)
ep = {}
for s, fid in EYEPACS_SPLIT_IDS.items():
    p = f'/content/eyepacs_split_{s}.csv'
    if not os.path.exists(p): dl(fid, p)
    ep[s] = pd.read_csv(p, dtype={'image_id': str}); ep[s]['source'] = 'eyepacs'
if not glob.glob('/content/img/**/*.jp*g', recursive=True) and not glob.glob('/content/img/**/*.png', recursive=True):
    for i, fid in enumerate(EYEPACS_IMAGE_ZIPS, 1):
        z = f'/content/eyepacs_batch{i}.zip'; print('downloading EyePACS batch', i); dl(fid, z)
        with zipfile.ZipFile(z) as zf: zf.extractall('/content/img')
        os.remove(z)
INDEX = {os.path.splitext(os.path.basename(p))[0]: p for p in glob.glob('/content/img/**/*.jp*g', recursive=True) + glob.glob('/content/img/**/*.png', recursive=True)}
# DDR
man = pd.read_csv(f'{SPLITS}/ddr_manifest_v2.csv', dtype={'image_id': str})
coh = pd.read_csv(f'{SPLITS}/ddr_split_intraset_v2.csv', dtype={'image_id': str})
d = coh[['image_id', 'group']].merge(man[['image_id', 'grade', 'source_split']], on='image_id', how='left')
assert d.grade.notna().all() and (d.grade <= 4).all()
d['level'] = d.grade.astype(int); d['binary_label'] = (d.level >= 2).astype(int); d['source'] = 'ddr'; src = d.source_split.str.lower()
ddr = {'train': d[src.str.startswith('train')].reset_index(drop=True), 'val': d[src.str.startswith('val')].reset_index(drop=True), 'test': d[src.str.startswith('test')].reset_index(drop=True)}
DDR_PNG = '/content/ddr_png'; os.makedirs(DDR_PNG, exist_ok=True)
if len(glob.glob(f'{DDR_PNG}/*.png')) < len(d):
    for z in sorted(glob.glob(f'{DDR_IN}/processed_v2/ddr_processed_batch*_v2.zip')):
        with zipfile.ZipFile(z) as zf: zf.extractall(DDR_PNG)
DDR_INDEX = {os.path.splitext(os.path.basename(p))[0]: p for p in glob.glob(f'{DDR_PNG}/*.png')}
assert not set(INDEX) & set(DDR_INDEX), 'image id collision between EyePACS and DDR'
INDEX.update(DDR_INDEX)
cols = ['image_id', 'level', 'binary_label', 'source']
splits = {s: pd.concat([ep[s][cols], ddr[s][cols]], ignore_index=True) for s in ('train', 'val')}
for s, df in splits.items():
    miss = set(df.image_id) - set(INDEX); assert not miss, f'{len(miss)} {s} images missing'
    print(f"pooled {s}: {len(df):,} images ({(df.source == 'eyepacs').sum():,} EyePACS, {(df.source == 'ddr').sum():,} DDR) | grades {df.level.value_counts().sort_index().to_dict()}")
# EyePACS fresh set
fresh = pd.read_csv(io.BytesIO(svc.files().get_media(fileId=find_id('eyepacs_split_fresh_v2.csv', F_SPLITS)).execute()), dtype={'image_id': str})
FRESH = '/content/fresh'; os.makedirs(FRESH, exist_ok=True)
if not glob.glob(f'{FRESH}/*.png'):
    for f_ in sorted(list_folder(F_FRESH_IMG), key=lambda x: x['name']):
        if not f_['name'].endswith('.zip'): continue
        z = f"/content/{f_['name']}"; print('downloading', f_['name']); dl(f_['id'], z)
        with zipfile.ZipFile(z) as zf: zf.extractall(FRESH)
        os.remove(z)
FRESH_INDEX = {os.path.splitext(n)[0]: f'{FRESH}/{n}' for n in os.listdir(FRESH) if n.endswith('.png')}
assert not set(fresh.image_id) - set(FRESH_INDEX), 'fresh images missing'
# BRSET (test only): extract the verified archive and preprocess as in BRSET_prep_inference; no labels read here
ACC = json.load(open(f'{BR}/brset_access_record.json'))
BLOCAL, BPNG = '/content/brset_local', '/content/brset_png'
for d_ in (BLOCAL, BPNG): os.makedirs(d_, exist_ok=True)
if not glob.glob(f'{BLOCAL}/**/fundus_photos', recursive=True):
    TAR = f"{BR}/{ACC['archive']}"; assert sha256_file(TAR) == ACC['archive_sha256'], 'BRSET archive differs from the access record'
    with tarfile.open(TAR) as t: t.extractall(BLOCAL)
photos = sorted(glob.glob(f'{BLOCAL}/**/fundus_photos/*', recursive=True)); BIDS = [os.path.splitext(os.path.basename(p))[0] for p in photos]
assert len(BIDS) == ACC['image_files']
from PIL import Image
from multiprocessing import Pool
def _brset_work(fp):
    out = f"{BPNG}/{os.path.splitext(os.path.basename(fp))[0]}.png"
    if os.path.exists(out) and os.path.getsize(out) > 0: return 'skip'
    try: Image.open(fp).convert('RGB').resize((512, 512), Image.LANCZOS).save(out); return 'ok'
    except Exception as e: return f'error: {e}'
with Pool(os.cpu_count()) as pool: res = list(pool.imap_unordered(_brset_work, photos, chunksize=32))
assert not [r for r in res if r.startswith('error')], 'BRSET preprocessing errors'
TEST_SETS = {'fresh': (list(fresh.image_id), FRESH_INDEX), 'ddrtest': (list(ddr['test'].image_id), DDR_INDEX), 'brset': (BIDS, {i: f'{BPNG}/{i}.png' for i in BIDS})}
print({k: len(v[0]) for k, v in TEST_SETS.items()}, 'test images ready')

downloading EyePACS batch 1
downloading EyePACS batch 2
downloading EyePACS batch 3
downloading EyePACS batch 4
downloading EyePACS batch 5
pooled train: 19,742 images (13,532 EyePACS, 6,210 DDR) | grades {0: 13044, 1: 1226, 2: 4291, 3: 458, 4: 723}
pooled val: 5,387 images (2,900 EyePACS, 2,487 DDR) | grades {0: 3374, 1: 324, 2: 1328, 3: 121, 4: 240}
downloading eyepacs_fresh_batch1_v2.zip
downloading eyepacs_fresh_batch2_v2.zip
downloading eyepacs_fresh_batch3_v2.zip
downloading eyepacs_fresh_batch4_v2.zip


/tmp/ipykernel_1722/743483071.py:50: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  with tarfile.open(TAR) as t: t.extractall(BLOCAL)


{'fresh': 15794, 'ddrtest': 3725, 'brset': 16266} test images ready


In [5]:
# Cell 4: dataset, builders, optimiser, training loop (canonical code, unchanged apart from data)
import timm
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from sklearn.metrics import roc_auc_score
from transformers import AutoModel, AutoConfig
torch.backends.cudnn.benchmark = True
torch.backends.cuda.matmul.allow_tf32 = True; torch.backends.cudnn.allow_tf32 = True
MEAN, STD = [0.485,0.456,0.406], [0.229,0.224,0.225]
TF_TRAIN = transforms.Compose([transforms.RandomResizedCrop(IMG_SIZE, scale=(0.8,1.0)),
    transforms.RandomHorizontalFlip(), transforms.RandomVerticalFlip(),
    transforms.ColorJitter(0.2,0.2,0.1), transforms.ToTensor(), transforms.Normalize(MEAN, STD)])
TF_EVAL = transforms.Compose([transforms.Resize(256), transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(), transforms.Normalize(MEAN, STD)])
class DR(Dataset):
    def __init__(self, df, tf): self.df = df.reset_index(drop=True); self.tf = tf
    def __len__(self): return len(self.df)
    def __getitem__(self, i):
        r = self.df.iloc[i]
        return self.tf(Image.open(INDEX[str(r['image_id'])]).convert('RGB')), int(r['level']), int(r['binary_label']), i
class ImgDS(Dataset):
    def __init__(self, ids, index): self.ids = list(ids); self.index = index
    def __len__(self): return len(self.ids)
    def __getitem__(self, i): return TF_EVAL(Image.open(self.index[self.ids[i]]).convert('RGB')), i
def seed_everything(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
def find_blocks(vit):
    best = None
    for _, mod in vit.named_modules():
        if isinstance(mod, nn.ModuleList) and len(mod) >= 12 and (best is None or len(mod) > len(best)): best = mod
    if best is None: raise RuntimeError('could not locate transformer blocks')
    return best
class RETFound(nn.Module):
    def __init__(self, num_classes, pretrained=True):
        super().__init__()
        self.base = AutoModel.from_pretrained(RETFOUND_REPO, add_pooling_layer=False) if pretrained else AutoModel.from_config(AutoConfig.from_pretrained(RETFOUND_REPO), add_pooling_layer=False)
        h = self.base.config.hidden_size
        self.fc_norm = nn.LayerNorm(h, eps=1e-6); self.classifier = nn.Linear(h, num_classes)
        nn.init.trunc_normal_(self.classifier.weight, std=0.01); nn.init.zeros_(self.classifier.bias)
    def forward(self, x):
        h = self.base(pixel_values=x).last_hidden_state
        return self.classifier(self.fc_norm(h[:, 1:].mean(dim=1)))
def build(cfg, pretrained=True):
    if cfg['arch'] == 'retfound':
        m = RETFound(cfg['nc'], pretrained); n = sum(p.numel() for p in m.base.parameters())
        if n < 2.9e8: raise RuntimeError(f'RETFound encoder only {n/1e6:.0f}M params, weights did not load')
        return m
    return timm.create_model(cfg['arch'], pretrained=pretrained, num_classes=cfg['nc'])
def make_optimizer(model, cfg):
    if cfg['arch'] != 'retfound':
        return torch.optim.AdamW(model.parameters(), lr=cfg['lr'], weight_decay=cfg['wd'])
    d_, lr = cfg['lldecay'], cfg['lr']; blocks = find_blocks(model.base); n = len(blocks)
    block_ids = {id(p) for blk in blocks for p in blk.parameters()}
    embed = [p for nm, p in model.base.named_parameters() if id(p) not in block_ids and 'embed' in nm.lower()]
    embed_ids = {id(p) for p in embed}
    rest = [p for p in model.base.parameters() if id(p) not in block_ids and id(p) not in embed_ids]
    groups = [{'params': embed, 'lr': lr * (d_ ** n)}]
    for i, blk in enumerate(blocks): groups.append({'params': list(blk.parameters()), 'lr': lr * (d_ ** (n - 1 - i))})
    groups.append({'params': rest + list(model.fc_norm.parameters()), 'lr': lr})
    groups.append({'params': list(model.classifier.parameters()), 'lr': lr * 10})
    return torch.optim.AdamW(groups, lr=lr, weight_decay=cfg['wd'])
def _probs(out): return torch.sigmoid(out[:, 0]) if out.shape[1] == 1 else torch.softmax(out, 1)[:, 2:].sum(1)
@torch.no_grad()
def predict(model, df, dev, bs=64):
    model.eval(); probs = np.zeros(len(df), dtype=np.float64)
    for x, _, _, idx in DataLoader(DR(df, TF_EVAL), batch_size=bs, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True):
        probs[idx.numpy()] = _probs(model(x.to(dev, non_blocking=True)).float()).double().cpu().numpy()
    return probs
@torch.no_grad()
def predict_ids(model, ids, index, dev, bs=64):
    model.eval(); probs = np.zeros(len(ids), dtype=np.float64)
    for x, idx in DataLoader(ImgDS(ids, index), batch_size=bs, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True):
        probs[idx.numpy()] = _probs(model(x.to(dev, non_blocking=True)).float()).double().cpu().numpy()
    return probs
def train_one(cfg, dev, seed, eval_bs):
    seed_everything(seed); tr, va = splits['train'], splits['val']
    model = build(cfg).to(dev)
    g = torch.Generator(); g.manual_seed(seed)
    dl_ = DataLoader(DR(tr, TF_TRAIN), batch_size=cfg['bs'], shuffle=True, num_workers=NUM_WORKERS, drop_last=True,
                     generator=g, pin_memory=True, persistent_workers=True, prefetch_factor=4)
    opt = make_optimizer(model, cfg); sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=cfg['epochs'])
    lossf = nn.BCEWithLogitsLoss() if cfg['nc'] == 1 else nn.CrossEntropyLoss(); scaler = torch.amp.GradScaler('cuda')
    best_auc, best_state, best_ep = -1.0, None, -1
    for ep_ in range(cfg['epochs']):
        model.train(); tot = 0.0; t0 = time.time()
        for x, lvl, lab, idx in dl_:
            x = x.to(dev, non_blocking=True); y = (lab.float() if cfg['nc'] == 1 else lvl.long()).to(dev, non_blocking=True)
            opt.zero_grad(set_to_none=True)
            with torch.amp.autocast('cuda'):
                out = model(x); loss = lossf(out[:, 0], y) if cfg['nc'] == 1 else lossf(out, y)
            scaler.scale(loss).backward()
            if cfg.get('clip'): scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(model.parameters(), cfg['clip'])
            scaler.step(opt); scaler.update(); tot += float(loss.item())
        sch.step()
        vp = predict(model, va, dev, eval_bs); vauc = float(roc_auc_score(va['binary_label'].values, vp)); star = ''
        if vauc > best_auc:
            best_auc, best_ep = vauc, ep_ + 1; best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}; star = '  <- best'
        print(f"  epoch {ep_+1}/{cfg['epochs']}  loss {tot/max(1,len(dl_)):.4f}  val_auc {vauc:.4f}  ({time.time()-t0:.0f}s){star}")
    model.load_state_dict(best_state); print(f'  restored best epoch {best_ep} (val_auc {best_auc:.4f})')
    return model, best_auc, best_ep
def load_ckpt(path, dev):
    blob = torch.load(path, map_location='cpu', weights_only=False); cfg = blob['config']
    m = build(cfg, pretrained=False); m.load_state_dict(blob['state_dict'], strict=True); return m.to(dev).eval(), cfg
print('ready')

ready


In [6]:
# Cell 5: train (resumable) and score every pooled model on the three test sets
dev = 'cuda'; RECORD_FILE = f'{AUDIT}/pooled_training_record_c1.json'
record = json.load(open(RECORD_FILE)) if os.path.exists(RECORD_FILE) else {'plan_sha256': PLAN_SHA, 'models': {}}
def pred_path(setname, name): return f'{PREDS}/' + ('brset_predictions_pooled_' if setname == 'brset' else f'pooled_predictions_{setname}_') + f'{name}_v2.csv'
for i, (mi, cfg, k) in enumerate(ORDER, 1):
    name = f"{cfg['name']}{tag(k)}"; ck = f'{CKPT}/pooled_{name}_v2.pt'; eval_bs = 32 if cfg['arch'] == 'retfound' else 64
    need = [s for s in TEST_SETS if not os.path.exists(pred_path(s, name))]
    if os.path.exists(ck) and not need: continue
    if os.path.exists(ck): model, _ = load_ckpt(ck, dev); print(f'[{i}/{len(ORDER)}] {name}: checkpoint found, scoring {need}')
    else:
        seed = BASE_SEED + 10 * mi + k; print(f'\n=== [{i}/{len(ORDER)}] {name} (seed {seed}) ==='); t0 = time.time()
        model, best_auc, best_ep = train_one(cfg, dev, seed, eval_bs)
        torch.save({'state_dict': model.state_dict(), 'config': cfg, 'seed': seed, 'best_val_auc': best_auc, 'best_epoch': best_ep,
                    'training_data': 'EyePACS v2 train + DDR official train (pooled)', 'plan_sha256': PLAN_SHA}, ck)
        record['models'][name] = {'seed': seed, 'best_val_auc': round(best_auc, 4), 'best_epoch': best_ep, 'minutes': round((time.time() - t0) / 60, 1), 'checkpoint_sha256': sha256_file(ck)}
    for s in need:
        ids, index = TEST_SETS[s]; pr = predict_ids(model, ids, index, dev, eval_bs)
        pd.DataFrame({'image_id': ids, 'predicted_probability': pr, 'model_name': cfg['name'], 'replicate': k, 'training_data': 'pooled'}).to_csv(pred_path(s, name), index=False, float_format='%.17g')
        record['models'].setdefault(name, {})[f'{s}_prediction_sha256'] = sha256_file(pred_path(s, name))
    with open(RECORD_FILE, 'w') as fh: json.dump(record, fh, indent=2)
    print(f'  {name}: done'); del model; torch.cuda.empty_cache()
# the DDR-trained models have never been scored on the EyePACS fresh set; do it now
for mi, cfg in enumerate(MODELS):
    for k in SEEDS:
        name = f"{cfg['name']}{tag(k)}"; out = f'{PREDS}/ddrtrained_predictions_fresh_{name}_v2.csv'
        if os.path.exists(out): continue
        model, _ = load_ckpt(f'{CKPT}/ddr_{name}_v2.pt', dev); ids, index = TEST_SETS['fresh']
        pr = predict_ids(model, ids, index, dev, 32 if cfg['arch'] == 'retfound' else 64)
        pd.DataFrame({'image_id': ids, 'predicted_probability': pr, 'model_name': cfg['name'], 'replicate': k, 'training_data': 'DDR'}).to_csv(out, index=False, float_format='%.17g')
        print('scored DDR-trained', name, 'on the fresh set'); del model; torch.cuda.empty_cache()
left = [f"{c['name']}{tag(k)}" for _, c, k in ORDER if not all(os.path.exists(pred_path(s, f"{c['name']}{tag(k)}")) for s in TEST_SETS)]
print('remaining pooled models:', left or 'none')


=== [6/12] efficientnet_b3_5class_seed2 (seed 20261113) ===


model.safetensors: reconstructing file:   0%|          |  0.00B / 49.3MB            

model.safetensors: downloading bytes:           |  0.00B            

  epoch 1/15  loss 0.9935  val_auc 0.9018  (48s)  <- best
  epoch 2/15  loss 0.6922  val_auc 0.9191  (21s)  <- best
  epoch 3/15  loss 0.6221  val_auc 0.9292  (22s)  <- best
  epoch 4/15  loss 0.5710  val_auc 0.9336  (21s)  <- best
  epoch 5/15  loss 0.5226  val_auc 0.9336  (22s)
  epoch 6/15  loss 0.4682  val_auc 0.9393  (22s)  <- best
  epoch 7/15  loss 0.4224  val_auc 0.9304  (21s)
  epoch 8/15  loss 0.3579  val_auc 0.9362  (22s)
  epoch 9/15  loss 0.3124  val_auc 0.9289  (22s)
  epoch 10/15  loss 0.2582  val_auc 0.9282  (22s)
  epoch 11/15  loss 0.2216  val_auc 0.9287  (21s)
  epoch 12/15  loss 0.1923  val_auc 0.9288  (22s)
  epoch 13/15  loss 0.1761  val_auc 0.9326  (22s)
  epoch 14/15  loss 0.1610  val_auc 0.9313  (22s)
  epoch 15/15  loss 0.1542  val_auc 0.9310  (22s)
  restored best epoch 6 (val_auc 0.9393)
  efficientnet_b3_5class_seed2: done

=== [7/12] resnet50_5class (seed 20261121) ===


model.safetensors: reconstructing file:   0%|          |  0.00B /  102MB            

...

  epoch 1/15  loss 0.7743  val_auc 0.9330  (106s)  <- best
  epoch 2/15  loss 0.6709  val_auc 0.9359  (105s)  <- best
  epoch 3/15  loss 0.6349  val_auc 0.9418  (105s)  <- best
  epoch 4/15  loss 0.6145  val_auc 0.9445  (105s)  <- best
  epoch 5/15  loss 0.5946  val_auc 0.9415  (105s)
  epoch 6/15  loss 0.5815  val_auc 0.9459  (105s)  <- best
  epoch 7/15  loss 0.5575  val_auc 0.9459  (105s)
  epoch 8/15  loss 0.5460  val_auc 0.9462  (105s)  <- best
  epoch 9/15  loss 0.5312  val_auc 0.9456  (105s)
  epoch 10/15  loss 0.5141  val_auc 0.9477  (106s)  <- best
  epoch 11/15  loss 0.5100  val_auc 0.9466  (105s)
  epoch 12/15  loss 0.4949  val_auc 0.9477  (106s)  <- best
  epoch 13/15  loss 0.4870  val_auc 0.9463  (105s)
  epoch 14/15  loss 0.4824  val_auc 0.9470  (105s)
  epoch 15/15  loss 0.4810  val_auc 0.9469  (105s)
  restored best epoch 12 (val_auc 0.9477)
  retfound_5class: done

=== [11/12] retfound_5class_seed1 (seed 20261132) ===


Loading weights:   0%|          | 0/390 [00:00<?, ?it/s]

  epoch 1/15  loss 0.7749  val_auc 0.9297  (107s)  <- best
  epoch 2/15  loss 0.6667  val_auc 0.9410  (106s)  <- best
  epoch 3/15  loss 0.6363  val_auc 0.9390  (105s)
  epoch 4/15  loss 0.6151  val_auc 0.9389  (105s)
  epoch 5/15  loss 0.5961  val_auc 0.9391  (105s)
  epoch 6/15  loss 0.5748  val_auc 0.9441  (105s)  <- best
  epoch 7/15  loss 0.5550  val_auc 0.9456  (106s)  <- best
  epoch 8/15  loss 0.5492  val_auc 0.9420  (105s)
  epoch 9/15  loss 0.5323  val_auc 0.9468  (105s)  <- best
  epoch 10/15  loss 0.5202  val_auc 0.9465  (106s)
  epoch 11/15  loss 0.5051  val_auc 0.9475  (105s)  <- best
  epoch 12/15  loss 0.4955  val_auc 0.9452  (105s)
  epoch 13/15  loss 0.4888  val_auc 0.9466  (105s)
  epoch 14/15  loss 0.4875  val_auc 0.9461  (105s)
  epoch 15/15  loss 0.4819  val_auc 0.9460  (105s)
  restored best epoch 11 (val_auc 0.9475)
  retfound_5class_seed1: done

=== [12/12] retfound_5class_seed2 (seed 20261133) ===


Loading weights:   0%|          | 0/390 [00:00<?, ?it/s]

  epoch 1/15  loss 0.7773  val_auc 0.9292  (106s)  <- best
  epoch 2/15  loss 0.6692  val_auc 0.9387  (105s)  <- best
  epoch 3/15  loss 0.6390  val_auc 0.9421  (105s)  <- best
  epoch 4/15  loss 0.6143  val_auc 0.9412  (105s)
  epoch 5/15  loss 0.5973  val_auc 0.9440  (105s)  <- best
  epoch 6/15  loss 0.5763  val_auc 0.9490  (105s)  <- best
  epoch 7/15  loss 0.5605  val_auc 0.9476  (105s)
  epoch 8/15  loss 0.5458  val_auc 0.9448  (105s)
  epoch 9/15  loss 0.5312  val_auc 0.9429  (105s)
  epoch 10/15  loss 0.5171  val_auc 0.9464  (105s)
  epoch 11/15  loss 0.5070  val_auc 0.9477  (105s)
  epoch 12/15  loss 0.4950  val_auc 0.9459  (105s)
  epoch 13/15  loss 0.4897  val_auc 0.9460  (105s)
  epoch 14/15  loss 0.4884  val_auc 0.9457  (105s)
  epoch 15/15  loss 0.4849  val_auc 0.9459  (105s)
  restored best epoch 6 (val_auc 0.9490)
  retfound_5class_seed2: done
scored DDR-trained densenet121_5class on the fresh set
scored DDR-trained densenet121_5class_seed1 on the fresh set
scored DDR-t

## Analysis
Runs once every pooled model is scored (it reports how many are present and refuses to run with fewer than all 12 unless `ALLOW_PARTIAL = True`, which is for a progress check only). Missed cases per 10,000 images at the same referral share; ties at the margin broken at random, scored as the expected value. Differences between arms use one shared set of 1,000 patient (DDR: filename-group) bootstrap resamples.

In [7]:
# Cell 6: analysis
ALLOW_PARTIAL = False
from scipy.stats import rankdata
CORE = [m['name'] for m in MODELS]
pooled_files = [pred_path(s, f'{a}{tag(k)}') for s in TEST_SETS for a in CORE for k in SEEDS]
present = [p for p in pooled_files if os.path.exists(p)]
assert all(os.path.getmtime(p) > PLAN_TIME for p in present), 'a pooled prediction predates the plan'
print(f'pooled prediction files present: {len(present)} of {len(pooled_files)}')
assert len(present) == len(pooled_files) or ALLOW_PARTIAL, 'not all pooled models are scored yet'
def rk(s): return rankdata(s) / len(s)
def pairscore(ra, rb): return np.maximum(ra, rb) + 1e-9 * np.minimum(ra, rb)
def miss_vec(score, y, f):
    """expected miss indicator per image when referring the top f (ties at the margin broken at random)"""
    n = len(score); R = int(round(f * n)); o = np.argsort(-score, kind='stable'); s_ = score[o]; thr = s_[R - 1]
    refer_p = np.where(score > thr, 1.0, 0.0); tie = score == thr; refer_p[tie] = (R - (score > thr).sum()) / tie.sum()
    return y * (1 - refer_p)
def boot_w(groups, B, seed):
    codes = pd.factorize(groups)[0]; G = codes.max() + 1; rng = np.random.default_rng(seed)
    W = np.empty((B + 1, len(codes)), np.float32); W[0] = 1.0
    for b in range(1, B + 1): W[b] = np.bincount(rng.integers(0, G, G), minlength=G)[codes]
    return W
def ci(x): x = np.asarray(x, float); return [float(x[0]), float(np.percentile(x[1:], 2.5)), float(np.percentile(x[1:], 97.5))]
def fmt(t): return f'{t[0]:.2f} ({t[1]:.2f} to {t[2]:.2f})'
def norm_ids(x): return pd.Series(x, dtype=str).str.replace(r'\.(jpe?g|png|tiff?)$', '', regex=True, flags=__import__('re').I).to_numpy()
def load_scores(paths, ids):
    d = pd.read_csv(paths, dtype={'image_id': str}); s = pd.Series(d.predicted_probability.values, index=norm_ids(d.image_id)).reindex(ids)
    assert s.notna().all(), paths; return s.to_numpy(np.float64)
SETS = {}
# BRSET labels: aggregates only
lb = pd.read_csv(f'{BR}/label_brset.csv', dtype=str); icol = [c for c in lb.columns if 'icdr' in c.lower()][0]; pcol = 'patient_id' if 'patient_id' in lb.columns else [c for c in lb.columns if 'patient' in c.lower()][0]
idcol = 'image_id' if 'image_id' in lb.columns else [c for c in lb.columns if 'image' in c.lower()][0]
lb['_k'] = norm_ids(lb[idcol]); lb = lb.set_index('_k'); g = pd.to_numeric(lb.loc[norm_ids(BIDS), icol], errors='coerce').to_numpy(); keep = np.isin(g, [0, 1, 2, 3, 4])
bids = norm_ids(BIDS)[keep]
SETS['BRSET'] = {'ids': bids, 'grade': g[keep].astype(int), 'group': lb.loc[bids, pcol].astype(str).to_numpy(),
                 'E': lambda a, k: f'{PREDS}/brset_predictions_eyepacs_{a}{tag(k)}_v2.csv', 'D': lambda a, k: f'{PREDS}/brset_predictions_ddr_{a}{tag(k)}_v2.csv', 'P': lambda a, k: pred_path('brset', f'{a}{tag(k)}')}
SETS['EyePACS fresh'] = {'ids': fresh.image_id.to_numpy(), 'grade': fresh['level'].astype(int).to_numpy(), 'group': fresh.patient_id.astype(str).to_numpy(), 'f': 0.25,
                 'E': lambda a, k: f'{PREDS}/eyepacs_predictions_fresh_{a}{tag(k)}_v2.csv', 'D': lambda a, k: f'{PREDS}/ddrtrained_predictions_fresh_{a}{tag(k)}_v2.csv', 'P': lambda a, k: pred_path('fresh', f'{a}{tag(k)}')}
dt = ddr['test']
SETS['DDR test'] = {'ids': dt.image_id.to_numpy(), 'grade': dt.level.to_numpy(), 'group': dt.group.astype(str).to_numpy(),
                 'E': lambda a, k: f'{PREDS}/ddr_predictions_{a}{tag(k)}_v2.csv', 'D': lambda a, k: f'{PREDS}/ddrtrained_predictions_test_{a}{tag(k)}_v2.csv', 'P': lambda a, k: pred_path('ddrtest', f'{a}{tag(k)}')}
for nm, S in SETS.items(): S.setdefault('f', round(1.31 * float((S['grade'] >= 2).mean()), 4)); print(f"{nm}: {len(S['grade']):,} images | {len(set(S['group'])):,} groups | f = {S['f']}")
OUT = {'plan_sha256': PLAN_SHA, 'pooled_files_present': len(present)}
for nm, S in SETS.items():
    W = boot_w(S['group'], 1000, 20261102); Wt = W.sum(1, dtype=np.float64)
    SC = {(src, a, k): load_scores(S[src](a, k), S['ids']) for src in ('E', 'D', 'P') for a in CORE for k in SEEDS if os.path.exists(S[src](a, k))}
    K = {m: rk(v) for m, v in SC.items()}
    for oname, y in (('severe', (S['grade'] >= 3).astype(float)), ('referable', (S['grade'] >= 2).astype(float)), ('pdr', (S['grade'] == 4).astype(float))):
        f = S['f']; single = {m: miss_vec(v, y, f) for m, v in SC.items()}
        def pair_arm(sa, sb):
            vs, bet = [], []
            for a, b in itertools.permutations(CORE, 2) if sa != sb else itertools.combinations(CORE, 2):
                for i in SEEDS:
                    for j in SEEDS:
                        ma, mb = (sa, a, i), (sb, b, j)
                        if ma in SC and mb in SC:
                            pv = miss_vec(pairscore(K[ma], K[mb]), y, f); vs.append(pv)
                            bet.append(np.minimum(W @ single[ma].astype(np.float32), W @ single[mb].astype(np.float32)) - W @ pv.astype(np.float32))
            return (np.mean(vs, 0) if vs else None), (np.mean(bet, 0) / Wt * 1e4 if bet else None), len(vs)
        arms = {}
        for src in ('E', 'D', 'P'):
            ms = [single[(src, a, k)] for a in CORE for k in SEEDS if (src, a, k) in single]
            if ms: arms[f'single_{src}'] = np.mean(ms, 0)
        arms['pair_mixed_E_D'], vs_better_mixed, n_mixed = pair_arm('E', 'D')
        arms['pair_pooled'], vs_better_pooled, n_pp = pair_arm('P', 'P')
        arms['pair_E'], vs_better_E, n_ee = pair_arm('E', 'E')
        rate = {k: ci(W @ v.astype(np.float32) / Wt * 1e4) for k, v in arms.items() if v is not None}
        def diff(a, b): return ci(W @ (arms[a] - arms[b]).astype(np.float32) / Wt * 1e4) if arms.get(a) is not None and arms.get(b) is not None else None
        # C1 matched design and run
        c1 = [single[('E', a, k)] - single[('P', a, k)] for a in CORE for k in SEEDS if ('E', a, k) in single and ('P', a, k) in single]
        res = {'missed_per_10k': rate,
               'C1_E_single_minus_pooled_single': ci(W @ np.mean(c1, 0).astype(np.float32) / Wt * 1e4) if c1 else None,
               'C1b_D_single_minus_pooled_single': diff('single_D', 'single_P'),
               'C2_mixed_pair_minus_pooled_single': diff('pair_mixed_E_D', 'single_P'),
               'C3_pooled_single_avg_minus_pooled_pair': diff('single_P', 'pair_pooled'),
               'C3b_pooled_pair_vs_better_member': ci(vs_better_pooled) if vs_better_pooled is not None else None,
               'C4_E_pair_minus_pooled_pair': diff('pair_E', 'pair_pooled'),
               'mixed_pair_vs_better_member': ci(vs_better_mixed) if vs_better_mixed is not None else None,
               'units': {'mixed_pairs': n_mixed, 'pooled_pairs': n_pp, 'E_pairs': n_ee}}
        OUT[f'{nm} | {oname}'] = res
        print(f"\n{nm} | {oname} (f = {f})  missed per 10,000: " + ', '.join(f'{k} {v[0]:.1f}' for k, v in rate.items()))
        for kk in ('C1_E_single_minus_pooled_single', 'C1b_D_single_minus_pooled_single', 'C2_mixed_pair_minus_pooled_single', 'C3_pooled_single_avg_minus_pooled_pair', 'C3b_pooled_pair_vs_better_member', 'C4_E_pair_minus_pooled_pair', 'mixed_pair_vs_better_member'):
            if res[kk]: print(f'   {kk:42s} {fmt(res[kk])}')
def verdict(t, sign=+1):
    if t is None: return 'not computed'
    lo, hi = t[1], t[2]
    return 'supported' if (sign > 0 and lo > 0) else ('contradicted' if (sign > 0 and hi < 0) else 'inconclusive')
P_ = OUT['BRSET | severe']
OUT['verdicts_brset_severe'] = {'C1 pooled beats EyePACS-only single': verdict(P_['C1_E_single_minus_pooled_single']), 'C2 pooled single beats mixed-source pair': verdict(P_['C2_mixed_pair_minus_pooled_single']),
                                'C3 pooled pair beats pooled average member': verdict(P_['C3_pooled_single_avg_minus_pooled_pair']), 'C4 pooled pair beats EyePACS-only pair': verdict(P_['C4_E_pair_minus_pooled_pair'])}
print('\nPre-specified predictions (BRSET, severe DR):'); [print(f'   {k}: {v}') for k, v in OUT['verdicts_brset_severe'].items()]
OUT['written_utc'] = datetime.datetime.now(datetime.timezone.utc).isoformat()
with open(f'{OUTD}/pooled_vs_pairing_c1.json', 'w') as fh: json.dump(OUT, fh, indent=1, default=float)
drive.flush_and_unmount(); drive.mount('/content/drive'); print('written', f'{OUTD}/pooled_vs_pairing_c1.json')

pooled prediction files present: 36 of 36
BRSET: 16,266 images | 8,524 groups | f = 0.0745
EyePACS fresh: 15,794 images | 7,897 groups | f = 0.25
DDR test: 3,725 images | 143 groups | f = 0.5943

BRSET | severe (f = 0.0745)  missed per 10,000: single_E 35.1, single_D 105.6, single_P 45.5, pair_mixed_E_D 45.7, pair_pooled 38.0, pair_E 25.2
   C1_E_single_minus_pooled_single            -10.35 (-15.03 to -5.79)
   C1b_D_single_minus_pooled_single           60.09 (50.89 to 69.56)
   C2_mixed_pair_minus_pooled_single          0.25 (-3.80 to 4.09)
   C3_pooled_single_avg_minus_pooled_pair     7.48 (4.78 to 10.18)
   C3b_pooled_pair_vs_better_member           1.22 (-2.05 to 3.44)
   C4_E_pair_minus_pooled_pair                -12.77 (-18.38 to -7.64)
   mixed_pair_vs_better_member                -11.26 (-14.56 to -7.85)

BRSET | referable (f = 0.0745)  missed per 10,000: single_E 138.3, single_D 260.3, single_P 141.8, pair_mixed_E_D 163.7, pair_pooled 126.4, pair_E 121.8
   C1_E_single_minus_p